# Vision-Language-Action Models

> **Placeholder.** This notebook will build a robot policy on a real open VLA checkpoint and a
> real robot dataset. The outline below fixes the scope, the theory and the findings; code comes
> later.

---

## Why this topic

A vision-language-action (VLA) model is a VLM whose output is not text but **robot actions**:
joint deltas or end-effector poses at 10–50 Hz. Pretraining on internet image–text gives
semantic generalisation ("pick up the extinct animal" → the toy dinosaur); robot data gives
control. The open questions of 2025–26 are how to represent continuous actions, how to run a
3B-parameter model fast enough for control, and how to improve from the robot's own experience.

## Theory

**Policy.** Given images $o_t$, proprioception $s_t$ and an instruction $\ell$, the VLA models
$\pi_\theta(A_t \mid o_t, s_t, \ell)$ over an **action chunk** $A_t = (a_t, \dots, a_{t+H-1})$:
predicting $H$ steps at once (action chunking) smooths control and amortises inference.

**Discrete actions (RT-2, OpenVLA).** Each action dimension is binned into 256 tokens and decoded
autoregressively with the LM head. Simple, but $H \times D$ tokens per chunk is slow.
**FAST** compresses a chunk with a DCT followed by byte-pair encoding, cutting the token count.

**Continuous actions (π0 family, GR00T).** A separate **action expert** transformer attends to the
VLM's KV cache and generates the chunk with **flow matching**: from noise
$A^{(0)} \sim \mathcal{N}(0, I)$, integrate $\tfrac{dA}{d\tau} = v_\theta(A^{(\tau)}, \tau \mid o, s, \ell)$
for a few steps, trained with $\lVert v_\theta(A^\tau) - (A - \epsilon) \rVert^2$ on
$A^\tau = \tau A + (1-\tau)\epsilon$. Continuous outputs at 50 Hz without tokenisation.

**Dual systems.** GR00T N1 and Gemini Robotics split a slow reasoning VLM (System 2: plan,
decompose, check success) from a fast action policy (System 1). Gemini Robotics-ER models the
System-2 side alone and calls VLAs as tools.

**Learning from experience.** Behaviour cloning caps at demonstrator quality; 2025 systems add
value functions and RL from the robot's own rollouts and human interventions.

## Findings from the past year

| Model | Date | What is new |
|---|---|---|
| **π0.5** (Physical Intelligence) | Apr 2025 | Co-training on web, multi-robot and high-level subtask data for open-world generalisation in unseen homes |
| **π\*0.6 / Recap** (Physical Intelligence) | late 2025 | RL from experience: a value function and advantage-conditioned policy trained on rollouts plus corrections |
| **GR00T N1 → N1.6/N1.7** (NVIDIA) | 2025–26 | Open humanoid foundation policy, dual-system, diffusion/flow action head; N2 preview adds world-model planning |
| **Gemini Robotics 1.5 / ER 1.5 / ER 1.6** (Google DeepMind) | Sep 2025 – Apr 2026 | VLA plus embodied-reasoning model; ER 1.6 improves pointing, counting, success detection and calls VLAs and Search as tools |
| **SmolVLA** (Hugging Face / LeRobot) | Jun 2025 | ~450M open VLA trained on community data; runs on a laptop GPU |
| **OpenVLA-OFT** | 2025 | Parallel decoding + continuous actions + chunking for fine-tuned OpenVLA |

## Planned chapters

| Chapter | What it will build |
|---|---|
| 1) Introduction | From VLM to VLA; discrete vs continuous actions; System 1/2 |
| 2) Setup | `lerobot/smolvla_base`-class checkpoint, a LeRobot dataset (e.g. SO-100 / LIBERO slice), simulator if available, `DEVICE` |
| 3) The math | Action chunking, binning vs FAST (DCT + BPE), flow-matching action expert, normalisation of actions/state |
| 4) From scratch | FAST tokenizer on real trajectories (compression ratio, reconstruction error); flow-matching sampler for an action chunk; temporal ensembling |
| 5) End-to-end | Fine-tune SmolVLA on a small task slice; open-loop action error and closed-loop success in simulation |
| 6) Variants | Chunk length $H$, number of flow steps, discrete vs continuous head |
| 7) Practical notes | Control-rate budgets, latency hiding (real-time chunking), safety and teleoperation data |
| 8) Conclusion | Decision table |

## Key references

- Brohan et al., *RT-2* (2023); Kim et al., *OpenVLA* (2024); Kim et al., *OpenVLA-OFT* (2025)
- Black et al., *π0: A Vision-Language-Action Flow Model for General Robot Control* (2024); Physical Intelligence, *π0.5* (2025), *π\*0.6* (2025)
- Pertsch et al., *FAST: Efficient Action Tokenization for VLAs* (2025)
- NVIDIA, *GR00T N1* (2025); Google DeepMind, *Gemini Robotics* (2025) and Gemini Robotics-ER 1.6 (Apr 2026)
- Shukor et al., *SmolVLA* (2025); Chi et al., *Diffusion Policy* (2023); Zhao et al., *ACT / ALOHA* (2023)